# handlers

> Workflow-specific alignment handlers: init, undo

In [ ]:
#| default_exp routes.alignment.handlers

In [ ]:
#| export
from typing import Any, Dict, List, Tuple
from pathlib import Path

from fasthtml.common import Div, Span, Button

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH
from cjm_workflow_state.history import pop_history

from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.core.models import VADChunk, WorkingSegment
from cjm_fasthtml_workflow_transcript_decomp.routes.models import AlignmentUrls, DecompUrls

# Alignment components
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.step_renderer import (
    render_align_toolbar, render_align_stats, render_align_column_body,
    render_align_footer_content, render_align_mini_stats_text,
)

# Decomposition components for shared chrome
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.step_renderer import (
    render_decomp_mini_stats_text, render_toolbar as render_decomp_toolbar,
    render_decomp_footer_content,
)

# Route core helpers
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.core import (
    _load_alignment_context, _update_alignment_state, _push_alignment_history,
    _to_vad_chunks, _build_card_stack_state,
    _get_decomp_segments, _update_decomp_segments, _get_decomp_focused_index,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.card_stack import (
    _build_nav_response,
)

# Debug flag for alignment data flow tracing (set False in production)
DEBUG_ALIGNMENT = True


## Mutation Response Builder

Builds the standard OOB response for alignment mutation handlers.
Includes card stack slots, progress, stats, toolbar, and mini-stats badges.

In [ ]:
#| export
def _build_mutation_response(
    chunk_dicts:List[Dict[str, Any]],  # Updated VAD chunks (serialized)
    focused_index:int,  # Updated focused chunk index
    visible_count:int,  # Visible card count
    history_depth:int,  # Current history depth (for undo button state)
    urls:AlignmentUrls,  # URL bundle
    is_auto_mode:bool=False,  # Whether card count is in auto-adjust mode
    segments:List[WorkingSegment]=None,  # Decomp segments (for cross-state mini-stats)
) -> Tuple:  # OOB response tuple
    """Build the standard OOB response for alignment mutation handlers."""
    cs_state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
    )
    chunks = _to_vad_chunks(chunk_dicts)

    # Navigation OOB (slots + progress + focus)
    nav_response = _build_nav_response(chunk_dicts, cs_state, urls)

    # Alignment stats OOB
    stats_oob = render_align_stats(chunks, oob=True)

    # Alignment toolbar OOB
    toolbar_oob = render_align_toolbar(
        undo_url=urls.undo,
        can_undo=(history_depth > 0),
        visible_count=visible_count,
        is_auto_mode=is_auto_mode,
        oob=True,
    )

    # Alignment mini-stats badge OOB
    mini_stats_oob = Span(
        render_align_mini_stats_text(chunks),
        id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
        hx_swap_oob="true"
    )

    result = (*nav_response, stats_oob, toolbar_oob, mini_stats_oob)

    # Cross-state: update decomp mini-stats if segments provided
    if segments is not None:
        decomp_mini_oob = Span(
            render_decomp_mini_stats_text(segments),
            id=StructureDecompHtmlIds.DECOMP_MINI_STATS,
            hx_swap_oob="true"
        )
        result = (*result, decomp_mini_oob)

    return result


## Initialize Handler

Fetches VAD data from the alignment service and stores in state.

In [ ]:
#| export
async def _handle_align_init(
    workflow:Any,  # StructureDecompWorkflow instance
    request:Any,  # FastHTML request object
    sess:Any,  # FastHTML session object
    urls:AlignmentUrls,  # URL bundle
    visible_count:int=5,  # Initial visible card count
    card_width:int=40,  # Initial card width in rem
) -> tuple:  # (column_body, mini_stats_oob)
    """Initialize alignment from audio file via VAD plugin.
    
    Note: KB system is always built by decomp init handler. This handler
    only returns column body and mini-stats OOB update.
    """
    session_id = get_session_id(sess)

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Starting alignment initialization")
        print(f"[ALIGN_INIT] session_id: {session_id}")

    # Get media_path from selected sources (Phase 1)
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    selected_sources = selection_state.get("selected_sources", [])

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] selected_sources count: {len(selected_sources)}")

    # Extract media_path from first selected source's source block
    media_path = None
    if selected_sources:
        first_source = selected_sources[0]
        block = workflow.source_service.get_transcription_by_id(
            record_id=first_source["record_id"],
            provider_id=first_source["provider_id"],
        )
        if DEBUG_ALIGNMENT and block:
            print(f"[ALIGN_INIT] block.media_path: {block.media_path}")
        if block:
            media_path = block.media_path

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] extracted media_path: {media_path}")

    # Fetch VAD data
    chunks = []
    audio_duration = 0.0
    if media_path and workflow.alignment_service.is_available():
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] Calling VAD plugin with media_path: {media_path}")
        chunks, audio_duration = await workflow.alignment_service.analyze_audio_async(media_path)
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] VAD returned {len(chunks)} chunks, duration: {audio_duration:.2f}s")

    # Serialize and store
    chunk_dicts = [c.to_dict() for c in chunks]
    _update_alignment_state(
        workflow, session_id,
        vad_chunks=chunk_dicts,
        focused_chunk_index=0,
        is_initialized=True,
        history=[],
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
        audio_duration=audio_duration,
    )

    # Render column body (Web Audio API handles accurate seeking)
    column_body = render_align_column_body(
        chunks=chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        urls=urls,
        kb_system=None,
        media_path=media_path,
    )

    # Mini-stats badge OOB update for the column header
    mini_stats_oob = Span(
        render_align_mini_stats_text(chunks),
        id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
        hx_swap_oob="true"
    )

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Returning column_body + mini_stats_oob")

    return (column_body, mini_stats_oob)


## Undo Handler

Restores previous alignment state from history and recalculates
decomposition segment timestamps from restored chunk assignments.

In [ ]:
#| export
def _handle_align_undo(
    workflow:Any,  # StructureDecompWorkflow instance
    request:Any,  # FastHTML request object
    sess:Any,  # FastHTML session object
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB response tuple
    """Undo the last alignment operation."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(workflow, session_id)

    if not ctx.history:
        return ()

    # Pop history
    snapshot, new_history = pop_history(ctx.history)
    restored_chunks = snapshot.get("vad_chunks", ctx.chunk_dicts)
    restored_focused = snapshot.get("focused_chunk_index", ctx.focused_chunk_index)

    # Recalculate decomp segment timestamps from restored chunk assignments
    segments = _get_decomp_segments(workflow, session_id)
    chunks = _to_vad_chunks(restored_chunks)

    # Clear all segment timestamps and rebuild from chunk assignments
    for s in segments:
        s.vad_chunk_indices = []
        s.start_time = None
        s.end_time = None

    for chunk in chunks:
        if chunk.assigned_segment is not None and chunk.assigned_segment < len(segments):
            seg = segments[chunk.assigned_segment]
            if chunk.index not in seg.vad_chunk_indices:
                seg.vad_chunk_indices.append(chunk.index)
                seg.vad_chunk_indices.sort()
            if seg.start_time is None or chunk.start_time < seg.start_time:
                seg.start_time = chunk.start_time
            if seg.end_time is None or chunk.end_time > seg.end_time:
                seg.end_time = chunk.end_time

    # Save both states
    _update_alignment_state(
        workflow, session_id,
        vad_chunks=restored_chunks,
        focused_chunk_index=restored_focused,
        history=new_history,
    )
    _update_decomp_segments(workflow, session_id, segments)

    return _build_mutation_response(
        chunk_dicts=restored_chunks,
        focused_index=restored_focused,
        visible_count=ctx.visible_count,
        history_depth=len(new_history),
        urls=urls,
        is_auto_mode=ctx.is_auto_mode,
        segments=segments,
    )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()